# Experiment 6: IOI Causal-Control Case Study

Proposal target: test whether low-uncertainty feature sets produce cleaner causal interventions. This notebook implements the first operational GPT-2 IOI patching scaffold:

1. build clean/corrupted IOI prompts;
2. collect layer-8 residual vectors at the answer position;
3. train or load a small VG-SAE on GPT-2 residual activations;
4. select feature patches by clean-corrupt latent differences;
5. compare logit-difference recovery and `U_patch = sum_i U_i`.

This is intentionally a scaffold for the causal benchmark. The real paper version should broaden prompts, seeds, and controls.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / "outputs" / ".matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))
DEVICE = torch.device("cpu")
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "notebooks"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from src.gpt2_activations import ActivationCacheConfig, cache_gpt2_residual_activations, load_activation_cache
from src.sae_evaluate import feature_uncertainty
from src.sae_model import VGSAEConfig, VariationalGarroteSAE
from src.sae_train import fit_sae

from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
layer = 8
cache_path = PROJECT_ROOT / "outputs" / "gpt2" / "gpt2_layer8_resid.pt"


In [ ]:
if not cache_path.exists():
    cache_gpt2_residual_activations(
        ActivationCacheConfig(model_name=model_name, layer=layer, max_tokens=2048, sequence_length=64),
        cache_path,
    )

train_x = load_activation_cache(cache_path)[:2048].to(DEVICE)
sae = VariationalGarroteSAE(
    VGSAEConfig(input_dim=train_x.shape[1], n_latents=2 * train_x.shape[1], lambda_sparsity=1.0, beta=1.0)
)
fit_sae(sae, train_x, max_steps=300, batch_size=128, lr=1e-3, history_every=100, seed=0)
U = feature_uncertainty(sae, train_x)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
lm = AutoModelForCausalLM.from_pretrained(model_name).to(DEVICE)
lm.eval()

clean_prompts = [
    "When John and Mary went to the store, John gave a bottle to",
    "When Alice and Bob visited the park, Alice handed a book to",
    "When Sarah and Tom left the office, Sarah passed the keys to",
]
corrupted_prompts = [
    "When Mary and John went to the store, John gave a bottle to",
    "When Bob and Alice visited the park, Alice handed a book to",
    "When Tom and Sarah left the office, Sarah passed the keys to",
]
correct_answers = [" Mary", " Bob", " Tom"]
incorrect_answers = [" John", " Alice", " Sarah"]
correct_ids = torch.tensor([tokenizer.encode(a, add_special_tokens=False)[0] for a in correct_answers], device=DEVICE)
incorrect_ids = torch.tensor([tokenizer.encode(a, add_special_tokens=False)[0] for a in incorrect_answers], device=DEVICE)


In [ ]:
def encode_prompts(prompts):
    return tokenizer(prompts, padding=True, return_tensors="pt").to(DEVICE)

def final_positions(encoded):
    return encoded["attention_mask"].sum(dim=1) - 1

def hidden_at_answer(prompts):
    encoded = encode_prompts(prompts)
    with torch.no_grad():
        outputs = lm(**encoded, output_hidden_states=True, use_cache=False)
    positions = final_positions(encoded)
    hidden = outputs.hidden_states[layer + 1][torch.arange(len(prompts)), positions]
    logits = outputs.logits[torch.arange(len(prompts)), positions]
    return hidden.detach(), logits.detach(), positions, encoded

def logit_diff(logits):
    return (logits[torch.arange(logits.shape[0]), correct_ids] - logits[torch.arange(logits.shape[0]), incorrect_ids]).detach()

clean_h, clean_logits, clean_pos, clean_encoded = hidden_at_answer(clean_prompts)
corrupt_h, corrupt_logits, corrupt_pos, corrupt_encoded = hidden_at_answer(corrupted_prompts)
print("clean logit diff", logit_diff(clean_logits))
print("corrupt logit diff", logit_diff(corrupt_logits))


In [ ]:
with torch.no_grad():
    clean_latents = sae(clean_h)["h"]
    corrupt_latents = sae(corrupt_h)["h"]
    delta = (clean_latents - corrupt_latents).abs().mean(dim=0)
    candidate_features = torch.topk(delta, k=32).indices

rows = []
for k in [2, 4, 8, 16, 32]:
    patch_features = candidate_features[:k]
    with torch.no_grad():
        corrupt_out = sae(corrupt_h)
        clean_out = sae(clean_h)
        patched_latents = corrupt_out["h"].clone()
        patched_latents[:, patch_features] = clean_out["h"][:, patch_features]
        patched_residual = sae.decode(patched_latents)
    rows.append({
        "k": k,
        "U_patch": float(U[patch_features].sum()),
        "patch_residual_norm": float((patched_residual - corrupt_h).norm(dim=1).mean()),
    })
patch_df = pd.DataFrame(rows)
patch_df


In [ ]:
def run_with_residual_patch(encoded, positions, patched_residual):
    batch_indices = torch.arange(patched_residual.shape[0], device=patched_residual.device)

    def hook(_module, _inputs, outputs):
        hidden = outputs[0].clone()
        hidden[batch_indices, positions] = patched_residual
        return (hidden,) + outputs[1:]

    handle = lm.transformer.h[layer].register_forward_hook(hook)
    try:
        with torch.no_grad():
            outputs = lm(**encoded, use_cache=False)
    finally:
        handle.remove()
    logits = outputs.logits[batch_indices, positions]
    return logits.detach()

rows = []
for k in [2, 4, 8, 16, 32]:
    patch_features = candidate_features[:k]
    with torch.no_grad():
        corrupt_out = sae(corrupt_h)
        clean_out = sae(clean_h)
        patched_latents = corrupt_out["h"].clone()
        patched_latents[:, patch_features] = clean_out["h"][:, patch_features]
        patched_residual = sae.decode(patched_latents)
    logits = run_with_residual_patch(corrupt_encoded, corrupt_pos, patched_residual)
    rows.append({
        "k": k,
        "U_patch": float(U[patch_features].sum()),
        "patched_logit_diff": float(logit_diff(logits).mean()),
        "clean_logit_diff": float(logit_diff(clean_logits).mean()),
        "corrupt_logit_diff": float(logit_diff(corrupt_logits).mean()),
    })
causal_df = pd.DataFrame(rows)
causal_df["recovery_fraction"] = (
    (causal_df["patched_logit_diff"] - causal_df["corrupt_logit_diff"])
    / (causal_df["clean_logit_diff"] - causal_df["corrupt_logit_diff"]).replace(0.0, np.nan)
)
causal_df


In [ ]:
out = OUTPUT_DIR / "exp06_ioi_causal_control"
out.mkdir(parents=True, exist_ok=True)
causal_df.to_csv(out / "ioi_causal_control.csv", index=False)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(causal_df["U_patch"], causal_df["recovery_fraction"])
for _, row in causal_df.iterrows():
    ax.annotate(f"k={int(row['k'])}", (row["U_patch"], row["recovery_fraction"]))
ax.set_xlabel("U_patch")
ax.set_ylabel("logit-diff recovery fraction")
ax.set_title("IOI patch reliability vs uncertainty")
fig.tight_layout()
fig.savefig(out / "ioi_causal_control.png", dpi=160)


**Critical caveat:** this is a first causal-control scaffold, not a publishable IOI benchmark. The publishable version needs more prompt templates, answer-token validation, negative controls, seed sweeps, activation-patching baselines, and comparison against TopK/Gated feature sets.
